# 🎓 Indic LLM Evaluation Lab: Malayalam QA with Qwen-2.5 3B & QLoRA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bilalvpm4321/malayalam-machine-learning/blob/main/notebooks/indic_llm_inference_server.ipynb)

**M.Tech Research Project**: Multilingual & Indic Large Language Model Evaluation and Improvement  
**Target Language**: Malayalam (`ml` / മലയാളം)  
**Base Model**: `Qwen/Qwen2.5-3B-Instruct`  
**Dataset**: AI4Bharat IndicQA Malayalam  
**Methodology**: 4-bit Quantized Low-Rank Adaptation (QLoRA, $r=16, \alpha=32$) with Assistant-Only Loss Masking  

---

### 📌 Pipeline Invariants
1. **Strict Maximum Sequence Length Assertion**: `assert len(input_ids) <= MAX_LENGTH` enforced on every single training sample.
2. **Question-Aware Inference Context Windowing**: If an inference passage exceeds the token budget, it identifies the highest question-keyword relevant window rather than blindly chopping the end.
3. **Zero Silent Fallback**: If `qlora` is requested when the adapter is unavailable, the API returns HTTP 503 / raises an exception rather than silently masquerading as base.
4. **Side-by-Side Research Benchmark**: Produces a comparative Base vs QLoRA evaluation table on the exact same test items.
5. **Complete Reproducibility**: Saves full hyperparameter metadata to `experiment_summary.json`.

## ⚙️ SECTION 1: Configuration & Experiment Flags `[CPU / SAFE]`

In [ ]:
import os
import time
import shutil
import glob
import re

# =============================================================================
# EXPERIMENT & PIPELINE CONFIGURATION FLAGS
# =============================================================================
RUN_TRAINING = False          # Set to True ONLY when you explicitly want to train
FORCE_RETRAIN = False         # Set to True to start fresh (archives/ignores existing checkpoints)
RUN_EVALUATION = False        # Run test set evaluation and comparison table
USE_QLORA_ADAPTER = False     # Load fine-tuned LoRA adapter if available (set True after training)
DEBUG_MODE = True             # True: Seeded debug subset (70 train / 13 test) | False: Full IndicQA (80/20)

# =============================================================================
# GOOGLE DRIVE PERSISTENT STORAGE CONFIGURATION
# =============================================================================
DRIVE_MOUNT_POINT = "/content/drive"
# Direct path requested for persistent QLoRA training storage
OUTPUT_DIR = "/content/drive/MyDrive/malayalam-qwen-qlora-chat" if (os.path.exists(DRIVE_MOUNT_POINT) or "/content" in os.getcwd()) else "./malayalam-qwen-qlora-chat"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
FINAL_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "final_adapter")
EXPERIMENTS_ARCHIVE_DIR = os.path.join(OUTPUT_DIR, "experiments")

# Minimum free Drive space required before training is allowed to start (in GB)
MIN_FREE_SPACE_GB = 5.0

# Checkpoint & Resume Configuration
RESUME_FROM_CHECKPOINT = True # Automatically detect and resume from latest valid checkpoint
SAVE_STRATEGY = "steps"
# Save every 10 steps in debug (creates ~5 checkpoints across 3 epochs); every 100 steps in full mode
SAVE_STEPS = 10 if DEBUG_MODE else 100
SAVE_TOTAL_LIMIT = 3          # Retain up to 3 most recent checkpoints to balance recovery & Drive quota
# Real-time live log frequency (every 1 step in debug for rich feedback; every 10 steps in full)
LOGGING_STEPS = 1 if DEBUG_MODE else 10
TEST_STOP_AFTER_STEPS = None  # Optional: Set to integer (e.g. 15) to simulate safe stop & test resume

# Model & Dataset Hyperparameters
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
DATASET_URL = "https://huggingface.co/datasets/ai4bharat/IndicQA/resolve/main/data/indicqa.ml.json"
MAX_LENGTH = 1536             # Max context + question + answer sequence length
MAX_NEW_TOKENS = 128          # Extractive QA generation limit (128 avoids UTF-8 multi-byte truncation)
SEED = 42

# QLoRA Hyperparameters
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training Hyperparameters (Optimized for Tesla T4 ~15GB VRAM)
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3 if DEBUG_MODE else 5

total_train_duration = None   # Tracks training duration
checkpoint_resume_info = {}   # Tracks checkpoint metadata for reproducibility

effective_batch_size = BATCH_SIZE * GRAD_ACCUM_STEPS
print("=" * 65)
print(f"🔧 CONFIGURATION SUMMARY:")
print(f"   • Base Model:             {MODEL_ID}")
print(f"   • Run Training:           {RUN_TRAINING}")
print(f"   • Force Retrain:          {FORCE_RETRAIN}")
print(f"   • Resume From Checkpoint: {RESUME_FROM_CHECKPOINT}")
print(f"   • Use QLoRA Adapter:      {USE_QLORA_ADAPTER}")
mode_label = "70 train / 13 test" if DEBUG_MODE else "Full 80/20 context-level split"
print(f"   • Debug Mode:             {DEBUG_MODE} ({mode_label})")
print(f"   • Batch Size / GradAccum: {BATCH_SIZE} / {GRAD_ACCUM_STEPS} (Effective Batch Size: {effective_batch_size})")
print(f"   • Output Directory:       {OUTPUT_DIR}")
print(f"   • Checkpoint Directory:   {CHECKPOINT_DIR}")
print(f"   • Min Free Drive Space:   {MIN_FREE_SPACE_GB} GB")
print(f"   • Save Strategy & Steps:  {SAVE_STRATEGY} (every {SAVE_STEPS} steps, max {SAVE_TOTAL_LIMIT} kept)")
print(f"   • Live Log Frequency:     Every {LOGGING_STEPS} step(s)")
print(f"   • Final Adapter Dir:      {FINAL_ADAPTER_DIR}")
print("=" * 65)


## 💾 SECTION 1.5: Google Drive Mount & Storage Management `[CPU / SAFE]`

- Mounts **Google Drive** to `/content/drive` for durable checkpoint persistence.
- Checks available Drive storage against `MIN_FREE_SPACE_GB`.
- Reports detailed project and checkpoint storage usage.

In [ ]:
# =============================================================================
# GOOGLE DRIVE MOUNT, STORAGE REPORT & SAFETY CHECKS
# =============================================================================
try:
    from google.colab import drive
    print("🔄 Mounting Google Drive to '/content/drive'...")
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully!")
except ImportError:
    print("ℹ️ Not running in Google Colab environment. Using local directory structure.")
except Exception as e:
    print(f"⚠️ Google Drive mount status: {e}")

# Create all required persistent directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)
os.makedirs(EXPERIMENTS_ARCHIVE_DIR, exist_ok=True)

# Utility functions for formatted storage reporting
def format_bytes(size_bytes: int) -> str:
    if size_bytes < 1024:
        return f"{size_bytes} B"
    elif size_bytes < 1024**2:
        return f"{size_bytes / 1024:.2f} KB"
    elif size_bytes < 1024**3:
        return f"{size_bytes / (1024**2):.2f} MB"
    else:
        return f"{size_bytes / (1024**3):.2f} GB"

def get_directory_size(path: str) -> int:
    total = 0
    if not os.path.exists(path):
        return 0
    for root, dirs, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            if not os.path.islink(fp):
                total += os.path.getsize(fp)
    return total

def get_drive_storage_info(path: str):
    check_path = path if os.path.exists(path) else (os.path.dirname(path) if os.path.exists(os.path.dirname(path)) else "/")
    total, used, free = shutil.disk_usage(check_path)
    return {
        "total_gb": total / (1024**3),
        "used_gb": used / (1024**3),
        "free_gb": free / (1024**3)
    }

def show_drive_storage(project_dir: str = OUTPUT_DIR):
    storage = get_drive_storage_info(project_dir)
    proj_size = get_directory_size(project_dir)
    ckpt_size = get_directory_size(CHECKPOINT_DIR)
    final_size = get_directory_size(FINAL_ADAPTER_DIR)
    
    print("\n" + "=" * 65)
    print("📊 GOOGLE DRIVE STORAGE REPORT")
    print("=" * 65)
    print(f"Google Drive storage check:")
    print(f"   • Total space:            {storage['total_gb']:.2f} GB")
    print(f"   • Used space:             {storage['used_gb']:.2f} GB")
    print(f"   • Free space:             {storage['free_gb']:.2f} GB")
    print("-" * 65)
    print(f"Project Breakdown ({project_dir}):")
    print(f"   • Project directory size: {format_bytes(proj_size)}")
    print(f"   • Checkpoint directory:   {format_bytes(ckpt_size)}")
    print(f"   • Final adapter size:     {format_bytes(final_size)}")
    
    # Subdirectories consuming space
    subdirs = []
    if os.path.exists(project_dir):
        for item in os.listdir(project_dir):
            item_path = os.path.join(project_dir, item)
            if os.path.isdir(item_path):
                subdirs.append((item, get_directory_size(item_path)))
    if subdirs:
        subdirs.sort(key=lambda x: x[1], reverse=True)
        print("-" * 65)
        print("📁 Top Space Consumers in Project:")
        for name, sz in subdirs:
            print(f"   • {name:<28} : {format_bytes(sz)}")
    print("=" * 65 + "\n")

# Display storage report
show_drive_storage(OUTPUT_DIR)

# Verify Drive is writable
def verify_drive_writable(target_dir: str) -> bool:
    test_path = os.path.join(target_dir, ".drive_write_test.tmp")
    try:
        with open(test_path, "w", encoding="utf-8") as f:
            f.write("write_verification_test")
        assert os.path.exists(test_path), "Drive write test file missing!"
        os.remove(test_path)
        return True
    except Exception as e:
        print(f"❌ ERROR: Drive directory '{target_dir}' is not writable: {e}")
        return False

drive_writable = verify_drive_writable(OUTPUT_DIR)
if "/content" in os.getcwd() and not drive_writable:
    raise RuntimeError("FATAL: Google Drive is not mounted or writable. Please mount Google Drive before training!")


## 🖥️ SECTION 2: GPU Environment & Memory Check `[CPU / SAFE]`

In [ ]:
import torch
import os

def print_gpu_status(stage="Current Status"):
    print(f"\n--- [GPU Status: {stage}] ---")
    if torch.cuda.is_available():
        device_name = torch.cuda.get_device_name(0)
        total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        alloc_mem = torch.cuda.memory_allocated(0) / (1024**3)
        res_mem = torch.cuda.memory_reserved(0) / (1024**3)
        print(f"GPU Name:            {device_name}")
        print(f"Total VRAM:          {total_mem:.2f} GB")
        print(f"Allocated VRAM:      {alloc_mem:.2f} GB")
        print(f"Reserved VRAM:       {res_mem:.2f} GB")
    else:
        print("⚠️ No GPU detected. Please go to Runtime -> Change runtime type -> T4 GPU!")

!nvidia-smi 2>/dev/null || echo 'nvidia-smi check complete'
print_gpu_status("Initial Baseline")

## 📦 SECTION 3: Install Required Dependencies `[CPU / SAFE]`

In [ ]:
!pip install -q --upgrade pip
!pip install -q transformers peft accelerate bitsandbytes datasets trl sentencepiece fastapi uvicorn pydantic nest-asyncio requests

print("✅ All dependencies successfully installed!")

## 📊 SECTION 4: Dataset Preparation, `answer_start` & Leakage-Free Splitting `[CPU / SAFE]`

In [ ]:
import json
import requests
import random
import numpy as np

rng = random.Random(SEED)
np.random.seed(SEED)

print(f"⬇️ Downloading IndicQA Malayalam dataset from: {DATASET_URL}...")
resp = requests.get(DATASET_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
resp.raise_for_status()
raw_json_data = resp.json()

# 1. Flatten records and preserve unique article/paragraph context_id
all_records = []
for art_idx, article in enumerate(raw_json_data.get("data", [])):
    for p_idx, paragraph in enumerate(article.get("paragraphs", [])):
        context_text = str(paragraph.get("context", "") or "").strip()
        unique_ctx_id = f"art_{art_idx}_p_{p_idx}"
        
        for q_idx, qa in enumerate(paragraph.get("qas", [])):
            qa_id = qa.get("id", f"indicqa_ml_{len(all_records)}")
            question_text = str(qa.get("question", "") or "").strip()
            answers = qa.get("answers", [])
            answer_text = ""
            answer_start = -1
            
            if answers and isinstance(answers, list) and len(answers) > 0:
                first_ans = answers[0] if isinstance(answers[0], dict) else {}
                answer_text = str(first_ans.get("text", "") or "").strip()
                raw_start = first_ans.get("answer_start")
                if raw_start is not None:
                    try:
                        answer_start = int(raw_start)
                    except (ValueError, TypeError):
                        answer_start = -1
            
            # Fallback search if answer_start was null or -1
            if answer_text and answer_start == -1:
                answer_start = context_text.find(answer_text)
            
            all_records.append({
                "id": str(qa_id),
                "context": context_text,
                "question": question_text,
                "answer": answer_text,
                "answer_start": int(answer_start) if isinstance(answer_start, (int, float)) and answer_start is not None else -1,
                "context_id": unique_ctx_id
            })

total_raw_count = len(all_records)

# 2. Quality Check & Data Cleaning
cleaned_records = []
seen_keys = set()
removed_empty_answers = 0
removed_empty_contexts = 0
removed_empty_questions = 0
removed_duplicates = 0

for r in all_records:
    if not r["answer"]:
        removed_empty_answers += 1
        continue
    if not r["context"]:
        removed_empty_contexts += 1
        continue
    if not r["question"]:
        removed_empty_questions += 1
        continue
    
    dedup_key = (r["context"], r["question"])
    if dedup_key in seen_keys:
        removed_duplicates += 1
        continue
    seen_keys.add(dedup_key)
    cleaned_records.append(r)

total_cleaned_count = len(cleaned_records)

# 3. Context-Level Splitting (80% Train, 20% Test) to prevent data leakage
unique_contexts = list(set(r["context_id"] for r in cleaned_records))
rng.shuffle(unique_contexts)

train_context_cutoff = int(0.80 * len(unique_contexts))
train_context_ids = set(unique_contexts[:train_context_cutoff])
test_context_ids = set(unique_contexts[train_context_cutoff:])

# Strict verification of zero context overlap
context_overlap = train_context_ids.intersection(test_context_ids)
assert len(context_overlap) == 0, f"FATAL: Context leakage: {len(context_overlap)} overlapping contexts!"

full_train_records = [r for r in cleaned_records if r["context_id"] in train_context_ids]
full_test_records = [r for r in cleaned_records if r["context_id"] in test_context_ids]

# 4. Apply Seeded Random Selection for Debug Mode
if DEBUG_MODE:
    train_records = rng.sample(full_train_records, min(70, len(full_train_records)))
    test_records = rng.sample(full_test_records, min(13, len(full_test_records)))
    split_note = f"DEBUG SUBSET (Seeded random: {len(train_records)} train / {len(test_records)} test)"
else:
    train_records = full_train_records
    test_records = full_test_records
    split_note = f"FULL DATASET ({len(train_records)} train / {len(test_records)} test)"

assert len(train_records) > 0, f"FATAL: train_records is empty! ({len(full_train_records)} full train records)"
assert len(test_records) > 0, f"FATAL: test_records is empty! ({len(full_test_records)} full test records)"

# 5. Statistics
context_lengths = [len(r["context"]) for r in train_records + test_records]
answer_lengths = [len(r["answer"]) for r in train_records + test_records]

print("=" * 65)
print(f"📊 DATASET QUALITY & SPLIT REPORT ({split_note}):")
print(f"   • Total Raw Records:           {total_raw_count}")
print(f"   • Clean Valid Records:         {total_cleaned_count}")
print(f"   • Total Unique Contexts:       {len(unique_contexts)}")
print(f"   • Context Leakage Overlap:     {len(context_overlap)} (0 = Strict Separation)")
print(f"   • Final Training Records:      {len(train_records)} (from {len(full_train_records)} pool)")
print(f"   • Final Testing Records:       {len(test_records)} (from {len(full_test_records)} pool)")
print(f"   • Avg Context Character Len:   {np.mean(context_lengths):.1f} chars (Max: {np.max(context_lengths)})")
print(f"   • Avg Answer Character Len:    {np.mean(answer_lengths):.1f} chars (Max: {np.max(answer_lengths)})")
print("=" * 65)

## 🎯 SECTION 5: Token-Budget Context Windowing & Assistant-Only Loss Masking `[CPU / SAFE]`

In [ ]:
from transformers import AutoTokenizer

LANGUAGE_SYSTEM_PROMPTS = {
    "ml": (
        "You are a helpful Malayalam Question Answering assistant. "
        "Please answer the question strictly in Malayalam using only the provided context. Give only the concise answer."
    ),
    "en": (
        "You are a helpful Question Answering assistant. "
        "Please answer the question clearly and accurately in English using only the provided context. Give only the concise answer."
    ),
    "ta": (
        "You are a helpful Tamil Question Answering assistant. "
        "Please answer the question strictly in Tamil using only the provided context. Give only the concise answer."
    ),
    "kn": (
        "You are a helpful Kannada Question Answering assistant. "
        "Please answer the question strictly in Kannada using only the provided context. Give only the concise answer."
    ),
    "hi": (
        "You are a helpful Hindi Question Answering assistant. "
        "Please answer the question strictly in Hindi using only the provided context. Give only the concise answer."
    ),
    "te": (
        "You are a helpful Telugu Question Answering assistant. "
        "Please answer the question strictly in Telugu using only the provided context. Give only the concise answer."
    ),
    "auto": (
        "You are a helpful multilingual Question Answering assistant. "
        "Detect the input language and answer the question in that exact same language using only the provided context. Give only the concise answer."
    )
}

def get_system_prompt_for_language(lang: str = "ml") -> str:
    normalized = (lang or "auto").lower().strip()
    return LANGUAGE_SYSTEM_PROMPTS.get(normalized, LANGUAGE_SYSTEM_PROMPTS["auto"])

SYSTEM_PROMPT = LANGUAGE_SYSTEM_PROMPTS["ml"]

print(f"📦 Loading tokenizer: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_qwen_chat_prompt(context: str, question: str, language: str = "ml") -> list:
    """Standard chat message structure for both training and multi-lingual inference."""
    sys_prompt = get_system_prompt_for_language(language)
    return [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion:\n{question}"}
    ]

def window_context_around_answer_once(context: str, answer: str, answer_start: int, max_chars: int = 2000) -> str:
    """
    Single-pass windowing centered on the ground-truth answer span using original offsets.
    """
    if len(context) <= max_chars:
        return context
    
    ans_start = answer_start if answer_start >= 0 else context.find(answer)
    if ans_start == -1:
        ans_start = 0
    
    ans_center = ans_start + (len(answer) // 2)
    half_window = max_chars // 2
    start_char = max(0, ans_center - half_window)
    end_char = min(len(context), start_char + max_chars)
    
    if end_char - start_char < max_chars:
        start_char = max(0, end_char - max_chars)
        
    return context[start_char:end_char].strip()

def window_context_for_inference(context: str, question: str, tokenizer, max_total_tokens: int = MAX_LENGTH, max_new_tokens: int = MAX_NEW_TOKENS, language: str = "ml") -> str:
    """
    Question-aware inference windowing: If context is long, locates sentences containing
    question keywords and centers the window around the highest-overlap section.
    """
    empty_msgs = format_qwen_chat_prompt("", question, language=language)
    overhead_str = tokenizer.apply_chat_template(empty_msgs, tokenize=False, add_generation_prompt=True)
    overhead_tokens = len(tokenizer.encode(overhead_str, add_special_tokens=False))
    
    available_ctx_tokens = max(100, max_total_tokens - overhead_tokens - max_new_tokens - 16)
    ctx_ids = tokenizer.encode(context, add_special_tokens=False)
    
    if len(ctx_ids) <= available_ctx_tokens:
        return context
    
    # Question keyword relevance localization
    q_words = set(question.split())
    sentences = context.split(". ")
    best_sent_idx = 0
    best_overlap = 0
    for idx, s in enumerate(sentences):
        s_words = set(s.split())
        overlap = len(q_words.intersection(s_words))
        if overlap > best_overlap:
            best_overlap = overlap
            best_sent_idx = idx
            
    # Join surrounding sentences centered at best_sent_idx
    start_idx = max(0, best_sent_idx - 1)
    end_idx = min(len(sentences), best_sent_idx + 3)
    candidate_text = ". ".join(sentences[start_idx:end_idx])
    candidate_ids = tokenizer.encode(candidate_text, add_special_tokens=False)
    
    if len(candidate_ids) <= available_ctx_tokens:
        return candidate_text
    
    # Fallback to direct truncate from beginning
    return tokenizer.decode(ctx_ids[:available_ctx_tokens], skip_special_tokens=True)

def prepare_training_sample(example, tokenizer, max_length=MAX_LENGTH):
    """
    Prepares training sample with assistant-only loss masking and guaranteed answer preservation.
    Guarantees len(input_ids) <= max_length.
    """
    raw_context = example["context"]
    question = example["question"]
    answer = example["answer"]
    ans_start = example.get("answer_start", -1)

    # 1. Tokenize answer with EOS first to reserve exact token budget
    answer_text = f"{answer}{tokenizer.eos_token}"
    answer_ids = tokenizer.encode(answer_text, add_special_tokens=False)
    
    # 2. Window context centered on answer in a single pass
    context = window_context_around_answer_once(raw_context, answer, ans_start, max_chars=1800)

    # 3. Format full chat template (defaulting to Malayalam for IndicQA training)
    prompt_messages = format_qwen_chat_prompt(context, question, language="ml")
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)

    # 4. If sequence exceeds max_length, shrink context window progressively while keeping answer centered
    curr_max_chars = 1800
    while len(prompt_ids) + len(answer_ids) > max_length and curr_max_chars > 300:
        curr_max_chars -= 250
        context = window_context_around_answer_once(raw_context, answer, ans_start, max_chars=curr_max_chars)
        prompt_messages = format_qwen_chat_prompt(context, question, language="ml")
        prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
        prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)

    input_ids = prompt_ids + answer_ids
    attention_mask = [1] * len(input_ids)
    labels = ([-100] * len(prompt_ids)) + answer_ids

    # Strict Invariant: Must not exceed MAX_LENGTH
    assert len(input_ids) <= max_length, (
        f"FATAL: Training sample exceeds MAX_LENGTH: {len(input_ids)} > {max_length}"
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "prompt_len": len(prompt_ids),
        "answer_len": len(answer_ids)
    }

# Verification on dataset sample
assert len(train_records) > 0, "Please run Section 4 first to populate train_records!"
sample_check = prepare_training_sample(train_records[0], tokenizer)
decoded_answer_from_labels = tokenizer.decode(
    [token for token in sample_check["labels"] if token != -100],
    skip_special_tokens=True
).strip()

print("=" * 65)
print("🔍 ASSISTANT-ONLY LOSS MASKING VERIFICATION:")
print(f"   • Total Tokens:               {len(sample_check['input_ids'])}")
print(f"   • Prompt Tokens (Masked -100):{sample_check['prompt_len']}")
print(f"   • Answer Tokens (Active Loss):{sample_check['answer_len']}")
print(f"   • Original Dataset Answer:    '{train_records[0]['answer']}'")
print(f"   • Decoded Loss Tokens:        '{decoded_answer_from_labels}'")
print("=" * 65)

assert sample_check["answer_len"] > 0, "FATAL: Answer token length is 0!"
assert decoded_answer_from_labels == train_records[0]["answer"].strip(), "FATAL: Decoded answer mismatch!"
print("✅ Masking & length assertion verified!")


## 🧪 SECTION 6: Pure CPU-Safe Unit Tests & Regression Verification `[CPU / SAFE]`

In [ ]:
from collections import Counter
import unicodedata
import re
import string

def normalize_indic_answer(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    text = text.strip().strip(string.punctuation + "।॥?,.!;:-")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def compute_qa_metrics(prediction: str, ground_truth: str) -> dict:
    norm_pred = normalize_indic_answer(prediction)
    norm_gold = normalize_indic_answer(ground_truth)
    if not norm_gold:
        return {"exact_match": None, "f1": None}
    em = 1 if norm_pred == norm_gold else 0
    pred_tokens = norm_pred.split()
    gold_tokens = norm_gold.split()
    if not pred_tokens or not gold_tokens:
        f1 = 1.0 if norm_pred == norm_gold else 0.0
        return {"exact_match": em, "f1": f1}
    common = sum((Counter(pred_tokens) & Counter(gold_tokens)).values())
    if common == 0:
        return {"exact_match": em, "f1": 0.0}
    precision = common / len(pred_tokens)
    recall = common / len(gold_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return {"exact_match": em, "f1": round(f1, 4)}

def run_cpu_safe_unit_tests():
    print("=" * 65)
    print("🧪 RUNNING PURE CPU-SAFE PIPELINE TESTS:")
    print("=" * 65)
    tests = []
    
    # 1. Zero Context Leakage Test
    overlap_count = len(train_context_ids.intersection(test_context_ids))
    tests.append(("Context Overlap == 0 (No Leakage)", overlap_count == 0))
    
    # 2. No Empty Answers in Splits
    no_empty = all(bool(r["answer"].strip()) for r in train_records + test_records)
    tests.append(("No Empty Answers in Train/Test", no_empty))
    
    # 3. Chat Template Structure Check
    msgs = format_qwen_chat_prompt("സന്ദർഭം", "ചോദ്യം")
    tests.append(("Chat Template 2-Turn Structure", len(msgs) == 2 and msgs[0]["role"] == "system"))
    
    # 4. Long Context with Answer Near the End Regression Test
    long_filler = "ഇത് കേരളത്തെക്കുറിച്ചുള്ള ഒരു വിശദമായ വിവരണമാണ്. " * 100
    target_answer = "തിരുവനന്തപുരം"
    long_context = f"{long_filler} കേരളത്തിന്റെ തലസ്ഥാനം {target_answer} ആകുന്നു."
    regression_sample = {
        "context": long_context,
        "question": "കേരളത്തിന്റെ തലസ്ഥാനം ഏതാണ്?",
        "answer": target_answer,
        "answer_start": long_context.find(target_answer)
    }
    reg_tokenized = prepare_training_sample(regression_sample, tokenizer, max_length=1536)
    decoded_reg_ans = tokenizer.decode([t for t in reg_tokenized["labels"] if t != -100], skip_special_tokens=True).strip()
    reg_pass = (decoded_reg_ans == target_answer) and (reg_tokenized["answer_len"] > 0) and (len(reg_tokenized["input_ids"]) <= 1536)
    tests.append(("Long Context Answer-at-End Preservation & Length <= 1536", reg_pass))
    
    # 5. True Multiset Counter F1 Test
    f1_res = compute_qa_metrics("തമിഴ് തെലുങ്ക് ഹിന്ദി", "തമിഴ് തെലുങ്ക് ഹിന്ദി")
    tests.append(("Counter Multiset F1 Computation", f1_res["exact_match"] == 1 and f1_res["f1"] == 1.0))
    
    # 6. Question-Aware Inference Context Windowing Test
    inf_ctx = window_context_for_inference(long_context, "കേരളത്തിന്റെ തലസ്ഥാനം ഏതാണ്?", tokenizer, max_total_tokens=1536)
    tests.append(("Question-Aware Inference Windowing Valid", target_answer in inf_ctx))
    
    all_passed = True
    for name, status in tests:
        status_str = "✅ PASS" if status else "❌ FAIL"
        if not status:
            all_passed = False
        print(f"   {status_str} : {name}")
    print("=" * 65)
    return all_passed

cpu_tests_passed = run_cpu_safe_unit_tests()
assert cpu_tests_passed, "FATAL: One or more CPU-safe pipeline tests failed!"

## 🤖 SECTION 7: Base Model Loading (4-Bit NF4 Quantization) `[⚠️ GPU / EXPENSIVE]`

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training
import gc

print_gpu_status("Before Model Loading")
print(f"📦 Loading {MODEL_ID} in 4-bit NF4 precision...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
base_model.config.use_cache = False

print_gpu_status("After 4-bit Base Model Loaded")
print("✅ Base Qwen 2.5 3B model loaded successfully on GPU!")

## 🚀 SECTION 8: QLoRA Fine-Tuning Pipeline `[⚠️ GPU / EXPENSIVE]`

- Executes **only when `RUN_TRAINING = True`**.
- Training wrapper is cleanly discarded after saving the adapter.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq, TrainerCallback
from datasets import Dataset
import time
import datetime
import shutil
import glob
import re
import math

# =============================================================================
# CHECKPOINT VALIDATION, RESUME & SAFE CLEANUP UTILITIES
# =============================================================================
def get_checkpoint_step(path: str) -> int:
    match = re.search(r"checkpoint-(\d+)", os.path.basename(path))
    return int(match.group(1)) if match else -1

def is_valid_checkpoint(ckpt_dir: str) -> bool:
    """Verifies that a checkpoint directory has valid state files and uncorrupted trainer_state."""
    if not os.path.isdir(ckpt_dir):
        return False
    trainer_state_path = os.path.join(ckpt_dir, "trainer_state.json")
    if not os.path.exists(trainer_state_path):
        return False
    try:
        with open(trainer_state_path, "r", encoding="utf-8") as f:
            state_data = json.load(f)
            if "global_step" not in state_data:
                return False
    except Exception:
        return False
    # Check for model/adapter weights or optimizer state
    has_weights = os.path.exists(os.path.join(ckpt_dir, "adapter_model.safetensors")) or \
                  os.path.exists(os.path.join(ckpt_dir, "adapter_model.bin")) or \
                  os.path.exists(os.path.join(ckpt_dir, "optimizer.pt"))
    return has_weights

def find_latest_valid_checkpoint(checkpoint_dir: str) -> str:
    """Scans checkpoint directory and returns the latest uncorrupted checkpoint path, or None."""
    if not os.path.exists(checkpoint_dir):
        return None
    candidates = glob.glob(os.path.join(checkpoint_dir, "checkpoint-*"))
    valid_ckpts = [p for p in candidates if is_valid_checkpoint(p)]
    if not valid_ckpts:
        return None
    valid_ckpts.sort(key=get_checkpoint_step, reverse=True)
    return valid_ckpts[0]

def verify_checkpoint_metadata_compatibility(ckpt_dir: str, current_debug_mode: bool) -> bool:
    """Ensures a DEBUG_MODE=True checkpoint is never used for a FULL training run or vice versa."""
    meta_file = os.path.join(ckpt_dir, "checkpoint_metadata.json")
    if os.path.exists(meta_file):
        try:
            with open(meta_file, "r", encoding="utf-8") as f:
                meta = json.load(f)
                if meta.get("debug_mode") != current_debug_mode:
                    print(f"⚠️ Incompatible Checkpoint: Created with DEBUG_MODE={meta.get('debug_mode')}, but current run is DEBUG_MODE={current_debug_mode}.")
                    return False
        except Exception:
            pass
    return True

def clean_old_checkpoints(checkpoint_dir: str, keep_count: int = 3, protect_checkpoint: str = None) -> int:
    """
    Safely removes obsolete, superseded checkpoints beyond keep_count to save Drive storage.
    NEVER deletes: the latest valid checkpoint, final_adapter/, or the checkpoint required for resume.
    """
    if not os.path.exists(checkpoint_dir):
        return 0
    ckpts = glob.glob(os.path.join(checkpoint_dir, "checkpoint-*"))
    if len(ckpts) <= keep_count:
        return 0
    ckpts.sort(key=get_checkpoint_step, reverse=True)
    protected = set()
    if ckpts:
        protected.add(os.path.abspath(ckpts[0]))
    if protect_checkpoint and os.path.exists(protect_checkpoint):
        protected.add(os.path.abspath(protect_checkpoint))
    for c in ckpts[:keep_count]:
        protected.add(os.path.abspath(c))
    
    removed_count = 0
    freed_bytes = 0
    for ck in ckpts[keep_count:]:
        abs_ck = os.path.abspath(ck)
        if abs_ck not in protected:
            sz = get_directory_size(ck)
            try:
                shutil.rmtree(ck)
                removed_count += 1
                freed_bytes += sz
            except Exception as e:
                print(f"   ⚠️ Could not remove {os.path.basename(ck)}: {e}")
    if removed_count > 0:
        print(f"🧹 Storage Cleanup: Removed {removed_count} obsolete checkpoint(s) (Freed {format_bytes(freed_bytes)}).")
    return removed_count

# =============================================================================
# REAL-TIME LIVE TRAINING PROGRESS & MONITORING CALLBACK
# =============================================================================
class LiveTrainingProgressCallback(TrainerCallback):
    """Displays real-time live metrics, loss, learning rate, GPU memory, ETA, and checkpoint events."""
    def __init__(self, total_epochs: int, total_steps: int, logging_steps: int = 1):
        self.total_epochs = total_epochs
        self.total_steps = total_steps
        self.logging_steps = logging_steps
        self.start_time = None
        self.last_logged_step = -1
    
    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        if state.max_steps and state.max_steps > 0:
            self.total_steps = state.max_steps
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or self.start_time is None:
            return
        step = state.global_step
        if step == self.last_logged_step:
            return
        self.last_logged_step = step
        
        # Metrics extraction
        loss = logs.get("loss")
        lr = logs.get("learning_rate")
        grad_norm = logs.get("grad_norm")
        epoch = state.epoch if state.epoch is not None else (step / max(1, self.total_steps) * self.total_epochs)
        
        # Timing & Speed
        now = time.time()
        elapsed_sec = max(0.1, now - self.start_time)
        pct = min(100.0, (step / max(1, self.total_steps)) * 100)
        speed = step / elapsed_sec if elapsed_sec > 0 else 0.0
        remaining_steps = max(0, self.total_steps - step)
        eta_sec = (remaining_steps / speed) if speed > 0 else 0.0
        
        elapsed_str = str(datetime.timedelta(seconds=int(elapsed_sec)))
        eta_str = str(datetime.timedelta(seconds=int(eta_sec)))
        
        # Lightweight GPU VRAM Check
        gpu_str = "CPU"
        if torch.cuda.is_available():
            alloc = torch.cuda.memory_allocated(0) / (1024**3)
            res = torch.cuda.memory_reserved(0) / (1024**3)
            tot = torch.cuda.get_device_properties(0).total_memory / (1024**3)
            gpu_str = f"Alloc: {alloc:.2f} GB | Res: {res:.2f} GB / {tot:.2f} GB"
        
        loss_val = f"{loss:.4f}" if loss is not None else "N/A"
        lr_val = f"{lr:.6e}" if lr is not None else "N/A"
        grad_val = f"{grad_norm:.2f}" if grad_norm is not None else "N/A"
        
        print(f"📊 [Step {step:4d}/{self.total_steps:4d} | Epoch {epoch:.2f}/{self.total_epochs} ({pct:5.1f}%)] " \
              f"Loss: {loss_val} | LR: {lr_val} | GradNorm: {grad_val} | " \
              f"Elapsed: {elapsed_str} | ETA: {eta_str} | Speed: {speed:.2f} st/s | GPU [{gpu_str}]")
    
    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        pct = min(100.0, (step / max(1, self.total_steps)) * 100)
        epoch = state.epoch if state.epoch is not None else 0.0
        ckpt_name = f"checkpoint-{step}"
        ckpt_path = os.path.join(args.output_dir, ckpt_name)
        print("
" + "=" * 65)
        print(f"💾 CHECKPOINT SAVED TO GOOGLE DRIVE")
        print("=" * 65)
        print(f"   • Checkpoint:        {ckpt_name}")
        print(f"   • Training Progress: {pct:.1f}%")
        print(f"   • Current Epoch:     {epoch:.2f} / {self.total_epochs}")
        print(f"   • Current Step:      {step} / {self.total_steps}")
        print(f"   • Drive Path:        {ckpt_path}")
        print("=" * 65 + "
")

class SimulatedStopCallback(TrainerCallback):
    """Allows testing checkpoint/resume safely without intentionally crashing the runtime."""
    def __init__(self, stop_at_step):
        self.stop_at_step = stop_at_step
    def on_step_end(self, args, state, control, **kwargs):
        if self.stop_at_step is not None and state.global_step >= self.stop_at_step:
            print(f"\n🛑 [SIMULATED STOP] Reached Step {state.global_step} >= {self.stop_at_step}. Stopping cleanly to verify resume.")
            control.should_training_stop = True

final_adapter_exists = os.path.exists(os.path.join(FINAL_ADAPTER_DIR, "adapter_config.json")) or \
                       os.path.exists(os.path.join(OUTPUT_DIR, "adapter_config.json"))

if RUN_TRAINING:
    # 1. Pre-Training Drive Storage Safety Check
    storage_info = get_drive_storage_info(OUTPUT_DIR)
    if storage_info["free_gb"] < MIN_FREE_SPACE_GB:
        show_drive_storage(OUTPUT_DIR)
        raise RuntimeError(
            f"❌ INSUFFICIENT DRIVE STORAGE: Free space is {storage_info['free_gb']:.2f} GB, "
            f"which is below the required safety threshold of {MIN_FREE_SPACE_GB} GB.\n"
            f"Please free up space in your Google Drive before starting training."
        )
    
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)
    os.makedirs(EXPERIMENTS_ARCHIVE_DIR, exist_ok=True)
    
    # 2. Determine resume checkpoint or safe archive
    resume_checkpoint = None
    resume_step_start = 0
    
    if FORCE_RETRAIN:
        print("⚠️ FORCE_RETRAIN=True: Starting a fresh experiment.")
        timestamp_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        run_archive_dir = os.path.join(EXPERIMENTS_ARCHIVE_DIR, f"run_{timestamp_str}")
        
        # Safely archive previous checkpoints
        existing_ckpts = glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint-*"))
        if existing_ckpts or os.path.exists(os.path.join(FINAL_ADAPTER_DIR, "adapter_config.json")):
            os.makedirs(run_archive_dir, exist_ok=True)
            if existing_ckpts:
                arch_ckpt_dir = os.path.join(run_archive_dir, "checkpoints")
                os.makedirs(arch_ckpt_dir, exist_ok=True)
                for ck in existing_ckpts:
                    shutil.move(ck, arch_ckpt_dir)
                print(f"📦 Safely archived {len(existing_ckpts)} old checkpoints to: {arch_ckpt_dir}")
            
            if os.path.exists(os.path.join(FINAL_ADAPTER_DIR, "adapter_config.json")):
                arch_adapt_dir = os.path.join(run_archive_dir, "final_adapter")
                shutil.copytree(FINAL_ADAPTER_DIR, arch_adapt_dir, dirs_exist_ok=True)
                print(f"📦 Safely archived previous final adapter to: {arch_adapt_dir}")
            
            old_summary = os.path.join(OUTPUT_DIR, "experiment_summary.json")
            if os.path.exists(old_summary):
                shutil.copy2(old_summary, os.path.join(run_archive_dir, "experiment_summary.json"))
    elif RESUME_FROM_CHECKPOINT:
        candidate_ckpt = find_latest_valid_checkpoint(CHECKPOINT_DIR)
        if candidate_ckpt and verify_checkpoint_metadata_compatibility(candidate_ckpt, DEBUG_MODE):
            resume_checkpoint = candidate_ckpt
            resume_step_start = get_checkpoint_step(resume_checkpoint)
            print(f"🎯 Valid Google Drive checkpoint found: {os.path.basename(resume_checkpoint)}")
        elif candidate_ckpt:
            print(f"⚠️ Checkpoint {os.path.basename(candidate_ckpt)} skipped due to configuration mismatch.")
        else:
            print("ℹ️ No existing checkpoint found on Google Drive. Starting fresh training run.")
    
    # Clean old checkpoints beyond limit before starting to save space
    clean_old_checkpoints(CHECKPOINT_DIR, keep_count=SAVE_TOTAL_LIMIT, protect_checkpoint=resume_checkpoint)
    
    # Calculate total steps expected
    num_samples = len(train_records)
    steps_per_epoch = math.ceil(num_samples / (BATCH_SIZE * GRAD_ACCUM_STEPS))
    total_expected_steps = steps_per_epoch * NUM_EPOCHS
    eff_batch = BATCH_SIZE * GRAD_ACCUM_STEPS
    gpu_device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    
    # 3. Rich Pre-Training Banner
    print("=" * 65)
    if resume_checkpoint:
        prev_pct = (resume_step_start / total_expected_steps * 100) if total_expected_steps > 0 else 0.0
        print(f"🔄 RESUMING QLoRA TRAINING FROM GOOGLE DRIVE")
        print(f"   • Status:                RESUMING FROM STEP {resume_step_start}")
        print(f"   • Latest Checkpoint:     {os.path.basename(resume_checkpoint)}")
        print(f"   • Previous Progress:     {prev_pct:.1f}%")
        print(f"   • Continuing training from Google Drive checkpoint...")
    else:
        print(f"🚀 QLoRA TRAINING STARTED (FRESH RUN)")
        print(f"   • Status:                STARTING FRESH")
        print(f"   • Existing Checkpoint:   None")
    print(f"   • Model:                 {MODEL_ID}")
    print(f"   • Dataset:               AI4Bharat IndicQA Malayalam")
    print(f"   • Mode:                  {'DEBUG (70 train / 13 test)' if DEBUG_MODE else 'FULL Dataset (80/20 split)'}")
    print(f"   • Training Samples:      {num_samples}")
    print(f"   • Evaluation Samples:    {len(test_records)}")
    print(f"   • Epochs:                {NUM_EPOCHS}")
    print(f"   • Batch Size / GradAcc:  {BATCH_SIZE} / {GRAD_ACCUM_STEPS} (Effective Batch Size: {eff_batch})")
    print(f"   • Total Planned Steps:   {total_expected_steps}")
    print(f"   • Learning Rate:         {LEARNING_RATE}")
    print(f"   • LoRA Rank / Alpha:     r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
    print(f"   • Checkpoint Save Dir:   {CHECKPOINT_DIR}")
    print(f"   • Checkpoint Save Freq:  Every {SAVE_STEPS} steps (Limit: {SAVE_TOTAL_LIMIT} kept in Drive)")
    print(f"   • Live Log Frequency:    Every {LOGGING_STEPS} step(s)")
    print(f"   • Free Drive Space:      {storage_info['free_gb']:.2f} GB (Min Required: {MIN_FREE_SPACE_GB} GB)")
    print(f"   • Hardware / GPU:        {gpu_device_name}")
    print("=" * 65)
    
    # 4. Model Preparation
    peft_train_model = prepare_model_for_kbit_training(base_model)
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM"
    )
    peft_train_model = get_peft_model(peft_train_model, lora_config)
    
    trainable_params, all_params = peft_train_model.get_nb_trainable_parameters()
    trainable_pct = (trainable_params / all_params) * 100
    print(f"📊 PARAMETER STATISTICS:")
    print(f"   • Trainable Parameters:  {trainable_params:,} ({trainable_params/1e6:.2f} M)")
    print(f"   • Total Parameters:      {all_params:,} ({all_params/1e9:.2f} B)")
    print(f"   • Trainable Percentage:  {trainable_pct:.4f} %")
    
    # 5. Prepare Tokenized Dataset
    tokenized_train_data = [prepare_training_sample(r, tokenizer) for r in train_records]
    hf_train_dataset = Dataset.from_list(tokenized_train_data)
    
    # 6. Training Arguments with Drive Checkpoint Saving (Directly on Drive, no duplicate /content bloat)
    training_args = TrainingArguments(
        output_dir=CHECKPOINT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        fp16=True,
        logging_steps=LOGGING_STEPS,
        save_strategy=SAVE_STRATEGY,
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        optim="paged_adamw_8bit",
        report_to="none",
        disable_tqdm=False,
        seed=SEED
    )
    
    live_progress_cb = LiveTrainingProgressCallback(
        total_epochs=NUM_EPOCHS,
        total_steps=total_expected_steps,
        logging_steps=LOGGING_STEPS
    )
    
    callbacks = [live_progress_cb]
    if TEST_STOP_AFTER_STEPS is not None:
        callbacks.append(SimulatedStopCallback(TEST_STOP_AFTER_STEPS))
    
    trainer = Trainer(
        model=peft_train_model,
        args=training_args,
        train_dataset=hf_train_dataset,
        data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True),
        callbacks=callbacks
    )
    
    # 7. Execute Training with Exception/Interruption Handling
    start_train_time = time.time()
    try:
        train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
        total_train_duration = time.time() - start_train_time
    except KeyboardInterrupt:
        latest_ck = find_latest_valid_checkpoint(CHECKPOINT_DIR)
        curr_storage = get_drive_storage_info(OUTPUT_DIR)
        print("
" + "=" * 65)
        print("⚠️ TRAINING INTERRUPTED BY USER")
        print("=" * 65)
        print(f"   • Last Completed Step:    {trainer.state.global_step}")
        print(f"   • Last Known Epoch:       {trainer.state.epoch:.2f}/{NUM_EPOCHS}")
        print(f"   • Latest Drive Checkpoint:{latest_ck}")
        print(f"   • Resume Available:       {'YES' if latest_ck else 'NO'}")
        print(f"   • Free Drive Space:       {curr_storage['free_gb']:.2f} GB")
        print("   • You can reconnect to Colab and resume training anytime from this Google Drive checkpoint.")
        print("   • Checkpoints have been preserved safely without automatic deletion.")
        print("=" * 65)
        raise
    except Exception as e:
        latest_ck = find_latest_valid_checkpoint(CHECKPOINT_DIR)
        curr_storage = get_drive_storage_info(OUTPUT_DIR)
        print("
" + "=" * 65)
        print("❌ TRAINING FAILED")
        print("=" * 65)
        print(f"   • Error:                  {e}")
        print(f"   • Last Completed Step:    {getattr(trainer.state, 'global_step', 'Unknown')}")
        print(f"   • Latest Checkpoint:      {os.path.basename(latest_ck) if latest_ck else 'None'}")
        print(f"   • Resume Available:       {'YES' if latest_ck else 'NO'}")
        print(f"   • Checkpoint Path:        {latest_ck or 'N/A'}")
        print(f"   • Free Drive Space:       {curr_storage['free_gb']:.2f} GB")
        print("   • The Drive checkpoint is safe and can be used to resume once the issue is resolved.")
        print("   • Checkpoints have been preserved safely without automatic deletion.")
        print("=" * 65)
        raise
    
    # Tag checkpoints with metadata for reproducibility
    for ck in glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint-*")):
        meta_p = os.path.join(ck, "checkpoint_metadata.json")
        if not os.path.exists(meta_p):
            with open(meta_p, "w", encoding="utf-8") as f:
                json.dump({"debug_mode": DEBUG_MODE, "model_id": MODEL_ID, "seed": SEED, "epochs": NUM_EPOCHS}, f)
    
    latest_final_ckpt = find_latest_valid_checkpoint(CHECKPOINT_DIR)
    
    # Clean old checkpoints beyond limit
    clean_old_checkpoints(CHECKPOINT_DIR, keep_count=SAVE_TOTAL_LIMIT, protect_checkpoint=latest_final_ckpt)
    
    # 8. Save Final LoRA Adapter & Tokenizer directly to Google Drive
    peft_train_model.save_pretrained(FINAL_ADAPTER_DIR)
    tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
    # Backwards compatibility in OUTPUT_DIR
    peft_train_model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    
    # 9. Save Comprehensive Experiment Metadata to Google Drive
    exp_suffix = "debug" if DEBUG_MODE else "full"
    duration_str = str(datetime.timedelta(seconds=int(total_train_duration)))
    timestamp_now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    checkpoint_resume_info = {
        "experiment_id": f"qwen2.5_3b_qlora_indicqa_ml_{exp_suffix}",
        "model_id": MODEL_ID,
        "dataset": "AI4Bharat IndicQA Malayalam",
        "mode": "DEBUG" if DEBUG_MODE else "FULL",
        "train_examples_count": len(train_records),
        "test_examples_count": len(test_records),
        "output_directory": OUTPUT_DIR,
        "checkpoint_directory": CHECKPOINT_DIR,
        "checkpoint_used_for_resume": resume_checkpoint,
        "was_resumed": bool(resume_checkpoint),
        "checkpoint_save_strategy": SAVE_STRATEGY,
        "save_steps": SAVE_STEPS,
        "save_total_limit": SAVE_TOTAL_LIMIT,
        "logging_steps": LOGGING_STEPS,
        "final_checkpoint": latest_final_ckpt,
        "final_adapter_directory": FINAL_ADAPTER_DIR,
        "final_loss": float(train_result.training_loss) if hasattr(train_result, "training_loss") and train_result.training_loss is not None else None,
        "total_train_duration_seconds": round(total_train_duration, 2),
        "total_train_duration_formatted": duration_str,
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
        "effective_batch_size": eff_batch,
        "learning_rate": LEARNING_RATE,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "lora_target_modules": LORA_TARGET_MODULES,
        "max_length": MAX_LENGTH,
        "max_new_tokens": MAX_NEW_TOKENS,
        "seed": SEED,
        "timestamp": timestamp_now
    }
    
    summary_path = os.path.join(OUTPUT_DIR, "experiment_summary.json")
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(checkpoint_resume_info, f, indent=2, ensure_ascii=False)
    
    # 10. Clean Training Completion Summary with Storage Report
    print("
" + "=" * 65)
    print("🎉 QLoRA TRAINING COMPLETED (SAVED TO GOOGLE DRIVE)")
    print("=" * 65)
    print(f"   • Status:                SUCCESS")
    print(f"   • Model:                 {MODEL_ID}")
    print(f"   • Dataset:               AI4Bharat IndicQA Malayalam")
    print(f"   • Training Mode:         {'DEBUG (70 examples)' if DEBUG_MODE else 'FULL Dataset'}")
    print(f"   • Training Examples:     {num_samples}")
    print(f"   • Epochs Completed:      {NUM_EPOCHS}/{NUM_EPOCHS}")
    print(f"   • Final Training Step:   {train_result.global_step}")
    print(f"   • Final Training Loss:   {train_result.training_loss:.4f}")
    print(f"   • Total Training Time:   {duration_str} ({total_train_duration:.1f}s)")
    print(f"   • Latest Checkpoint:     {latest_final_ckpt}")
    print(f"   • Final Adapter Path:    {FINAL_ADAPTER_DIR}")
    print(f"   • Metadata Summary:      {summary_path}")
    print("=" * 65)
    print("✅ Training completed successfully. The final LoRA adapter is ready on Google Drive for evaluation.\n")
    
    # Show updated storage usage
    show_drive_storage(OUTPUT_DIR)
    
    # 11. Clean up training wrapper GPU memory
    del peft_train_model
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    print_gpu_status("After Training Cleanup")
elif final_adapter_exists:
    print(f"ℹ️ Saved QLoRA adapter found in Google Drive at '{FINAL_ADAPTER_DIR}'. Training skipped (RUN_TRAINING = {RUN_TRAINING}).")
    show_drive_storage(OUTPUT_DIR)
else:
    print(f"ℹ️ RUN_TRAINING is set to False and no adapter exists yet. Using Base Model.")


## 🔌 SECTION 9: QLoRA Adapter Loading & Strict Dual-Config Context `[⚠️ GPU / EXPENSIVE]`

In [ ]:
from peft import PeftModel
from contextlib import contextmanager

active_model = base_model
qlora_loaded = False

adapter_load_path = FINAL_ADAPTER_DIR if os.path.exists(os.path.join(FINAL_ADAPTER_DIR, "adapter_config.json")) else OUTPUT_DIR
adapter_file_present = os.path.exists(os.path.join(adapter_load_path, "adapter_config.json"))

if USE_QLORA_ADAPTER and adapter_file_present:
    print(f"📦 Loading saved QLoRA adapter from '{adapter_load_path}' onto base model...")
    try:
        active_model = PeftModel.from_pretrained(base_model, adapter_load_path)
        qlora_loaded = True
        print("✅ QLoRA adapter loaded and ready for inference!")
    except Exception as e:
        qlora_loaded = False
        print(f"❌ ERROR: Failed to load QLoRA adapter: {e}")
        raise RuntimeError(f"Adapter load failed: {e}")
else:
    print("ℹ️ Running with Base Model (QLoRA adapter not loaded).")

@contextmanager
def model_inference_context(config_name: str):
    """
    Strict, zero-overhead switcher between Base model and QLoRA adapter in memory.
    Guarantees that QLoRA NEVER silently falls back to Base.
    """
    c_name = config_name.lower().strip()
    if c_name == "qlora":
        if not qlora_loaded or active_model == base_model:
            raise RuntimeError("QLoRA configuration requested, but adapter is not loaded.")
        yield active_model
    elif c_name == "base":
        if qlora_loaded:
            with active_model.disable_adapter():
                yield active_model
        else:
            yield base_model
    else:
        raise ValueError(f"Unknown configuration '{config_name}'. Must be 'base' or 'qlora'.")

print_gpu_status("Dual-Model Inference Ready")


## 🧪 SECTION 10: Model Sanity Checks & Side-by-Side Benchmark `[MODEL CHECK / GPU]`

In [ ]:
def run_single_inference(context: str, question: str, config: str = "base", max_new_tokens: int = MAX_NEW_TOKENS, language: str = "ml") -> str:
    windowed_context = window_context_for_inference(context, question, tokenizer, max_total_tokens=MAX_LENGTH, max_new_tokens=max_new_tokens, language=language)
    messages = format_qwen_chat_prompt(windowed_context, question, language=language)
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(base_model.device)
    
    with torch.inference_mode():
        with model_inference_context(config) as m:
            outputs = m.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
    response_tokens = outputs[0][inputs.input_ids.shape[1]:]
    answer = tokenizer.decode(response_tokens, skip_special_tokens=True).strip()
    
    tokens_count = len(response_tokens)
    has_ufffd = "\ufffd" in answer
    print(f"[INFERENCE DEBUG] Config: {config} | max_new_tokens: {max_new_tokens} | Generated Tokens: {tokens_count}")
    print(f"[INFERENCE DEBUG] repr(answer): {repr(answer)}")
    print(f"[INFERENCE DEBUG] Contains U+FFFD (): {has_ufffd}")
    return answer

def evaluate_dataset_split(records, config_name: str):
    em_count = 0
    f1_total = 0.0
    latencies = []
    
    for item in records:
        t0 = time.time()
        pred = run_single_inference(item["context"], item["question"], config=config_name)
        latencies.append((time.time() - t0) * 1000)
        metrics = compute_qa_metrics(pred, item["answer"])
        em_count += (metrics["exact_match"] or 0)
        f1_total += (metrics["f1"] or 0.0)
    
    em_pct = (em_count / len(records)) * 100
    f1_pct = (f1_total / len(records)) * 100
    avg_lat = float(np.mean(latencies))
    return round(em_pct, 2), round(f1_pct, 2), round(avg_lat, 1)

if RUN_EVALUATION:
    print("=" * 65)
    print("🧪 1. MANUAL MALAYALAM SANITY TESTS (Base vs QLoRA):")
    
    for cfg in (["base", "qlora"] if qlora_loaded else ["base"]):
        print(f"\n--- Running Sanity on [{cfg.upper()}] ---")
        ans = run_single_inference(
            context="മെഷീൻ ലേണിങ് എന്നത് ആർട്ടിഫിഷ്യൽ ഇന്റലിജൻസിന്റെ ഒരു ശാഖയാണ്. ഡാറ്റയിൽ നിന്ന് മാതൃകകൾ പഠിച്ച് പ്രവചനങ്ങൾ നടത്താൻ കമ്പ്യൂട്ടറുകളെ സഹായിക്കുന്ന സാങ്കേതികവിദ്യയാണ് മെഷീൻ ലേണിങ്.",
            question="മെഷീൻ ലേണിങ് എന്താണ്?",
            config=cfg,
            max_new_tokens=MAX_NEW_TOKENS
        )
        print(f"[{cfg.upper()} Answer]: {ans}")
    
    print("\n" + "=" * 65)
    print("📊 2. SIDE-BY-SIDE BENCHMARK ON TEST DATASET:")
    print("=" * 65)
    eval_rows = []
    for cfg in (["base", "qlora"] if qlora_loaded else ["base"]):
        em, f1, lat = evaluate_dataset_split(test_records, cfg)
        eval_rows.append({
            "Configuration": cfg.upper(),
            "Exact Match (%)": f"{em:.1f}%",
            "F1 Score (%)": f"{f1:.1f}%",
            "Avg Latency (ms)": f"{lat:.0f} ms",
            "Status": "Evaluated"
        })
    
    header = f"| {'Configuration':<15} | {'Exact Match (%)':<16} | {'F1 Score (%)':<14} | {'Avg Latency (ms)':<18} | {'Status':<10} |"
    divider = f"|{'-'*17}|{'-'*18}|{'-'*16}|{'-'*20}|{'-'*12}|"
    print(header)
    print(divider)
    for row in eval_rows:
        print(f"| {row['Configuration']:<15} | {row['Exact Match (%)']:<16} | {row['F1 Score (%)']:<14} | {row['Avg Latency (ms)']:<18} | {row['Status']:<10} |")


## 🌐 SECTION 11: FastAPI REST Inference Server `[CPU / SAFE]`

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import time

app = FastAPI(title="Indic LLM Evaluation Server", version="2.5.0")

# Explicit CORS origins for local Vite frontend
app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:5174",
        "http://localhost:5173",
        "http://127.0.0.1:5174",
        "http://127.0.0.1:5173",
        "*"
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class PredictionRequest(BaseModel):
    language: str = "ml"
    task: str = "qa"
    configuration: str = "base"  # "base" | "qlora"
    context: str
    question: str
    expected_answer: Optional[str] = ""
    max_new_tokens: Optional[int] = 128

@app.get("/health")
def health_check():
    return {
        "status": "ok",
        "model": MODEL_ID,
        "supported_configurations": ["base", "qlora"],
        "active_configuration": "qlora" if qlora_loaded else "base",
        "qlora_adapter_loaded": qlora_loaded,
        "max_new_tokens_default": MAX_NEW_TOKENS,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "gpu_memory_used_gb": round(torch.cuda.memory_allocated(0) / (1024**3), 2) if torch.cuda.is_available() else 0,
        "version": "2.5.0"
    }

@app.post("/predict")
def predict(req: PredictionRequest):
    t0 = time.time()
    config_choice = req.configuration.lower().strip() if req.configuration else "base"
    gen_tokens_limit = req.max_new_tokens if req.max_new_tokens and req.max_new_tokens > 0 else MAX_NEW_TOKENS
    
    # Strict validation
    if config_choice not in ["base", "qlora"]:
        raise HTTPException(
            status_code=400,
            detail=f"Invalid configuration '{req.configuration}'. Supported: ['base', 'qlora']"
        )
    
    if config_choice == "qlora" and not qlora_loaded:
        raise HTTPException(
            status_code=503,
            detail="QLoRA adapter is not loaded on this server. Please train/load the adapter first."
        )
    
    # Question-aware context windowing for inference
    windowed_context = window_context_for_inference(req.context, req.question, tokenizer, max_total_tokens=MAX_LENGTH, max_new_tokens=gen_tokens_limit, language=req.language)
    messages = format_qwen_chat_prompt(windowed_context, req.question, language=req.language)
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(prompt_text, return_tensors="pt").to(base_model.device)
    
    with torch.inference_mode():
        with model_inference_context(config_choice) as m:
            outputs = m.generate(
                **inputs,
                max_new_tokens=gen_tokens_limit,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
    
    response_tokens = outputs[0][inputs.input_ids.shape[1]:]
    generated_answer = tokenizer.decode(response_tokens, skip_special_tokens=True).strip()
    latency_ms = int((time.time() - t0) * 1000)
    
    tokens_count = len(response_tokens)
    has_ufffd = "\ufffd" in generated_answer
    
    # Transparent Research / Debug Logging (No modification to generated answer)
    print(f"[FASTAPI /predict DEBUG] Lang: '{req.language}' | Config: '{config_choice}' | max_new_tokens: {gen_tokens_limit} | Tokens: {tokens_count}")
    print(f"[FASTAPI /predict DEBUG] repr(answer): {repr(generated_answer)}")
    print(f"[FASTAPI /predict DEBUG] Contains U+FFFD (): {has_ufffd}")
    
    # Compute Evaluation Metrics
    metrics = compute_qa_metrics(generated_answer, req.expected_answer or "")
    
    return {
        "answer": generated_answer,
        "expected_answer": req.expected_answer or None,
        "exact_match": metrics["exact_match"],
        "f1": metrics["f1"],
        "similarity": metrics["f1"] if metrics["f1"] is not None else (1.0 if metrics["exact_match"] == 1 else 0.0),
        "latency_ms": latency_ms,
        "tokens_generated": tokens_count,
        "contains_replacement_char": has_ufffd,
        "model": MODEL_ID,
        "configuration": config_choice,
        "language": req.language
    }

print("✅ FastAPI endpoints created: GET /health, POST /predict")


## 🚀 SECTION 12: Cloudflare Quick Tunnel & Server Startup `[CPU / SAFE]`

In [ ]:
import os
import subprocess
import time
import uvicorn
import threading
import re

# 1. Clean up any existing cloudflared processes
!pkill -f cloudflared 2>/dev/null || true
time.sleep(1)

# 2. Download Cloudflare tunnel binary if missing
if not os.path.exists("cloudflared"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# 3. Launch Cloudflare tunnel
log_file = open("tunnel.log", "w")
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"], stdout=log_file, stderr=log_file)

# 4. Poll tunnel.log until URL appears
tunnel_url = None
print("⏳ Establishing Cloudflare Tunnel connection...")
for _ in range(30):
    time.sleep(1)
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r") as f:
            matches = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', f.read())
            if matches:
                tunnel_url = matches[-1]
                break

# 5. Start Uvicorn in background thread
def start_uvicorn():
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="warning")
    server = uvicorn.Server(config)
    server.run()

server_thread = threading.Thread(target=start_uvicorn, daemon=True)
server_thread.start()

print("=" * 65)
if tunnel_url:
    print("🎉 YOUR ACTIVE PUBLIC TUNNEL URL IS:")
    print(f"👉 {tunnel_url}")
    print("\nCopy this URL and paste it into your local Web App Settings page!")
else:
    print("⚠️ Tunnel started. Check tunnel.log if the URL didn't print within 30s.")
print("=" * 65)